用于信号生成、过滤和处理的高性能计算模块。

# Generation

## generate_nb

使用信号选择函数 `choice_func_nb` 生成 `bool` 数组。
- `choice_func_nb` 的函数签名：func(from_i, to_i, col, *args) -> np.array
  - 参数
    - from_i：搜索范围起始索引(包含)
    - to_i：搜索范围结束索引(不包含)
    - col：当前列索引
    - *args：传递给选择函数的额外参数
  - 返回值：索引数组，范围在[from_i, to_i)内
  - 该函数根据 `from_i, to_i, col` 生成序号表示行索引，这些行索引处须为 `True`
- 该函数按如下方式使用信号选择函数
  ```python
  for col in range(out.shape[1]):
    idxs = choice_func_nb(0, shape[0], col, *args)
  ```
  

参数
- `shape` (tuple)：目标信号矩阵的形状，格式为(行数, 列数)
- `pick_first` (bool)：是否只选择choice_func_nb返回的第一个信号
  - True：只取第一个信号，用于单次信号生成
  - False：取所有信号，用于多次信号生成
- `choice_func_nb` (callable)：信号选择函数，必须是Numba编译的函数
- `*args`：传递给 choice_func_nb 的额外参数

返回值：np.array，形状为 shape 的布尔数组，True 表示信号位置

### 源码

```python
@njit
def generate_nb(shape: tp.Shape,
                pick_first: bool,
                choice_func_nb: tp.ChoiceFunc, *args) -> tp.Array2d:

    out = np.full(shape, False, dtype=np.bool_)

    for col in range(out.shape[1]):
        idxs = choice_func_nb(0, shape[0], col, *args)
        if len(idxs) == 0:
            continue
        if pick_first:
            first_i = idxs[0]
            if first_i < 0 or first_i >= shape[0]:
                raise ValueError("First returned index is out of bounds")
            out[first_i, col] = True
        else:
            if np.any(idxs < 0) or np.any(idxs >= shape[0]):
                raise ValueError("Returned indices are out of bounds")
            out[idxs, col] = True
    return out
```

### 例子

In [1]:
import numpy as np
from numba import njit
from vectorbt.signals.nb import generate_nb

@njit
def choice_func_nb(from_i, to_i, col):
    # 每列在不同位置生成信号
    return np.array([from_i + col])

# 生成3列5行的信号矩阵
signals = generate_nb((5, 3), False, choice_func_nb)
print(signals)

[[ True False False]
 [False  True False]
 [False False  True]
 [False False False]
 [False False False]]


## generate_ex_nb

为每个入场信号生成对应的退出信号。

（1）遍历每一列，找到每一列的所有入场信号
- 对于 `entries` $E \in {\left\{ {bool} \right\}^{T \times N}}$ 的每一列 $n$（每个资产）
- 找到所有入场信号的索引集合 ${1_n} = \left\{ {{t_1},{t_2}, \cdots ,{t_K}} \right\}$，其中 ${E_{{t_{k,n}}}} = true$

（2）对于每个入场信号，确定其出场信号的搜索区间
- 对于每个入场信号 $t_k$
  - 起点为 $fro{m_k} = {t_k} + wait$
  - 终点：如果 `until_next = True` 并且存在下一个入场信号 $t_{k+1}$，则 $to_k=t_{k+1}$；否则 $to_k=T$

（3）确定出场信号
- 令 $O_k$ = `exit_choice_func_nb`$\left( {fro{m_k},t{o_k},n} \right)$
- 如果 `pick_first = True`，只取 $O_k$ 的第一个索引 ${O_{k,1}}$，令 ${X_{{O_{k,1}},n}} = True$；否则 ${X_{{O_k},n}} = True$

注意：如果 `skip_until_exit=True`，那么下一个入场信号必须在上一个出场信号之后才会被处理。

### 源码

```python
@njit
def generate_ex_nb(entries: tp.Array2d,
                   wait: int,
                   until_next: bool,
                   skip_until_exit: bool,
                   pick_first: bool,
                   exit_choice_func_nb: tp.ChoiceFunc, *args) -> tp.Array2d:

    exits = np.full_like(entries, False)

    for col in range(entries.shape[1]):
        entry_idxs = np.flatnonzero(entries[:, col])
        last_exit_i = -1
        for i in range(entry_idxs.shape[0]):
            # Calculate the range to choose from
            if skip_until_exit and entry_idxs[i] <= last_exit_i:
                continue
            from_i = entry_idxs[i] + wait
            if i < entry_idxs.shape[0] - 1 and until_next:
                to_i = entry_idxs[i + 1]
            else:
                to_i = entries.shape[0]
            if to_i > from_i:
                # Run the UDF
                idxs = exit_choice_func_nb(from_i, to_i, col, *args)
                if len(idxs) == 0:
                    continue
                if pick_first:
                    first_i = idxs[0]
                    if first_i < from_i or first_i >= to_i:
                        raise ValueError("First returned index is out of bounds")
                    exits[first_i, col] = True
                    last_exit_i = first_i
                else:
                    if np.any(idxs < from_i) or np.any(idxs >= to_i):
                        raise ValueError("Returned indices are out of bounds")
                    exits[idxs, col] = True
                    last_exit_i = idxs[-1]
    return exits
```

### 例子

In [ ]:
import numpy as np
from numba import njit
from vectorbt.signals.nb import generate_ex_nb

@njit
def exit_after_3_bars(from_i, to_i, col):
    if from_i + 2 < to_i:
        return np.array([from_i + 2], dtype=np.int64)
    return np.empty(0, dtype=np.int64)  # 推荐用 np.empty 而不是 np.array([])

# 你的 entries
entries = np.array([[True, False], [False, False], 
                    [False, False], [False, False]])
exits = generate_ex_nb(entries, 1, True, False, True, 
                        exit_after_3_bars)
print(f"入场信号\n{entries}")
print(f"退出信号\n{exits}")

## generate_enex_nb
生成入场信号和退出信号。

流程：

遍历每一列 $n$（时间×资产，$T \times N$）
- `prev_i` 和 `prev_prev_i` 分别记录上一个、上上一个 `True` 索引
- 处理顺序：入场——>出场——>入场——>出场——>...
  - 处理入场时
    - `from_i = prev_i + entry_wait`，第一次处理时，`from_i=0`
    - `to_i =` $T$
    - 调用 `idxs = entry_choice_func_nb(from_i, to_i, col, *entry_args)`
      - 即在 `from_i~to_i` 行内，`col` 列，选出应该为 `True` 的行索引
  - 处理出场时
    - `from_i = prev_i + exit_wait`
    - `to_i =` $T$
    - 调用 `idxs = exit_choice_func_nb(from_i, to_i, col, *exit_args)`
      - 即在 `from_i~to_i` 行内，`col` 列，选出应该为 `True` 的行索引
  - 根据 `entry_pick_first/exit_pick_first=True/False`，置 `entries/exits[idxs[0]/idxs, col]=True`
  - 每次处理完一次入/出场，根据 `entry_pick_first/exit_pick_first`
    - `True`
      - `entries/exits[idxs[0], col]=True`
      - `prev_prev_i = prev_i`，`prev_i = idxs[0]`
    - `False`
      - `entries/exits[idxs, col]=True`
      - `prev_prev_i = prev_i`，`prev_i = idxs[-1]`

### 源码

```python
@njit
def generate_enex_nb(shape: tp.Shape,
                     entry_wait: int,
                     exit_wait: int,
                     entry_pick_first: bool,
                     exit_pick_first: bool,
                     entry_choice_func_nb: tp.ChoiceFunc,
                     entry_args: tp.Args,
                     exit_choice_func_nb: tp.ChoiceFunc,
                     exit_args: tp.Args) -> tp.Tuple[tp.Array2d, tp.Array2d]:

    entries = np.full(shape, False)
    exits = np.full(shape, False)
    if entry_wait == 0 and exit_wait == 0:
        raise ValueError("entry_wait and exit_wait cannot be both 0")

    for col in range(shape[1]):
        prev_prev_i = -2
        prev_i = -1
        i = 0
        while True:
            to_i = shape[0]
            # Cannot assign two functions to a var in numba
            if i % 2 == 0:
                if i == 0:
                    from_i = 0
                else:
                    from_i = prev_i + entry_wait
                if from_i >= to_i:
                    break
                idxs = entry_choice_func_nb(from_i, to_i, col, *entry_args)
                a = entries
                pick_first = entry_pick_first
            else:
                from_i = prev_i + exit_wait
                if from_i >= to_i:
                    break
                idxs = exit_choice_func_nb(from_i, to_i, col, *exit_args)
                a = exits
                pick_first = exit_pick_first
            if len(idxs) == 0:
                break
            first_i = idxs[0]
            if first_i == prev_i == prev_prev_i:
                raise ValueError("Infinite loop detected")
            if first_i < from_i:
                raise ValueError("First index is out of bounds")
            if pick_first:
                # Consider only the first signal
                if first_i >= to_i:
                    raise ValueError("First index is out of bounds")
                a[first_i, col] = True
                prev_prev_i = prev_i
                prev_i = first_i
                i += 1
            else:
                # Consider all signals
                last_i = idxs[-1]
                if last_i >= to_i:
                    raise ValueError("Last index is out of bounds")
                a[idxs, col] = True
                prev_prev_i = prev_i
                prev_i = last_i
                i += 1

    return entries, exits
```

### 例子

In [ ]:
import numpy as np
import pandas as pd
from numba import njit
from vectorbt.signals.nb import generate_enex_nb

# 1. 生成模拟数据
np.random.seed(42)  # 设置随机种子，确保结果可重现
n_steps = 100
n_assets = 3

# 生成价格数据（随机游走）
price_data = np.zeros((n_steps, n_assets))
price_data[0] = 100  # 初始价格

for i in range(1, n_steps):
    # 随机收益率
    returns = np.random.normal(0, 0.02, n_assets)
    price_data[i] = price_data[i-1] * (1 + returns)

# 2. 计算移动平均线
def calculate_ma(prices, window):
    """计算移动平均线"""
    ma = np.zeros_like(prices)
    for col in range(prices.shape[1]):
        for i in range(prices.shape[0]):
            if i < window - 1:
                ma[i, col] = np.nan
            else:
                ma[i, col] = np.mean(prices[i-window+1:i+1, col])
    return ma

# 计算快线和慢线
ma_fast = calculate_ma(price_data, window=5)   # 5日移动平均线
ma_slow = calculate_ma(price_data, window=20)  # 20日移动平均线

# 3. 定义入场和出场函数
@njit
def ma_cross_entry(from_i, to_i, col, ma_fast, ma_slow):
    """快线上穿慢线入场"""
    for i in range(from_i, to_i):
        # 确保有足够的历史数据
        if i > 0 and not (np.isnan(ma_fast[i, col]) or np.isnan(ma_slow[i, col]) or 
                          np.isnan(ma_fast[i-1, col]) or np.isnan(ma_slow[i-1, col])):
            # 快线上穿慢线：当前快线>慢线，且前一个时刻快线<=慢线
            if ma_fast[i, col] > ma_slow[i, col] and ma_fast[i-1, col] <= ma_slow[i-1, col]:
                return np.array([i], dtype=np.int64)
    return np.empty(0, dtype=np.int64)

@njit  
def ma_cross_exit(from_i, to_i, col, ma_fast, ma_slow):
    """快线下穿慢线退出"""
    for i in range(from_i, to_i):
        # 确保有足够的历史数据
        if i > 0 and not (np.isnan(ma_fast[i, col]) or np.isnan(ma_slow[i, col]) or 
                          np.isnan(ma_fast[i-1, col]) or np.isnan(ma_slow[i-1, col])):
            # 快线下穿慢线：当前快线<慢线，且前一个时刻快线>=慢线
            if ma_fast[i, col] < ma_slow[i, col] and ma_fast[i-1, col] >= ma_slow[i-1, col]:
                return np.array([i], dtype=np.int64)
    return np.empty(0, dtype=np.int64)

# 4. 生成信号
entries, exits = generate_enex_nb(
    shape=(n_steps, n_assets),
    entry_wait=1, exit_wait=1,
    entry_pick_first=True, exit_pick_first=True,
    entry_choice_func_nb=ma_cross_entry,
    entry_args=(ma_fast, ma_slow),
    exit_choice_func_nb=ma_cross_exit,
    exit_args=(ma_fast, ma_slow)
)

# 5. 结果分析和可视化
print("=== 信号生成结果 ===")
print(f"数据形状: {entries.shape}")
print(f"入场信号总数: {entries.sum()}")
print(f"出场信号总数: {exits.sum()}")

# 为每个资产显示信号分布
for col in range(n_assets):
    entry_count = entries[:, col].sum()
    exit_count = exits[:, col].sum()
    print(f"资产 {col}: 入场信号 {entry_count} 个, 出场信号 {exit_count} 个")

# 6. 创建DataFrame便于查看
df = pd.DataFrame({
    'price': price_data[:, 0],
    'ma_fast': ma_fast[:, 0],
    'ma_slow': ma_slow[:, 0],
    'entry_signal': entries[:, 0],
    'exit_signal': exits[:, 0]
})

print("\n=== 前20行数据预览 ===")
print(df.head(50))


# Filtering

## clean_enex_1d_nb
清理一维入场和退出信号数组：
- 初始状态 + 入场信号（`entries[i]=True`） → 入场状态并且 `entries_out[i]=True`
- 初始状态 + 退出信号（`exits[i]=True`） → 退出状态并且 `exits_out[i]=True` [仅当 `entry_first=False`]
- 入场状态 + 退出信号（`exits[i]=True`） → 退出状态并且 `exits_out[i]=True`
- 退出状态 + 入场信号（`entries[i]=True`） → 入场状态并且 `entries_out[i]=True`
- 同时出现入场和退出信号 → 忽略该位置

### 源码
```python
@njit(cache=True)
def clean_enex_1d_nb(entries: tp.Array1d,
                     exits: tp.Array1d,
                     entry_first: bool) -> tp.Tuple[tp.Array1d, tp.Array1d]:

    entries_out = np.full(entries.shape, False, dtype=np.bool_)
    exits_out = np.full(exits.shape, False, dtype=np.bool_)

    phase = -1
    for i in range(entries.shape[0]):
        if entries[i] and exits[i]:
            continue
        if entries[i]:
            if phase == -1 or phase == 0:
                phase = 1
                entries_out[i] = True
        if exits[i]:
            if (not entry_first and phase == -1) or phase == 1:
                phase = 0
                exits_out[i] = True

    return entries_out, exits_out
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import clean_enex_1d_nb

entries = np.array([True, False, True, True, True, False])
exits = np.array([False, True, True, False, False, False])

# 清理信号，要求入场优先
clean_entries, clean_exits = clean_enex_1d_nb(entries, exits, True)

print(f"入场信号\n{entries}")
print(f"退出信号\n{exits}")
print(f"清理后的入场信号\n{clean_entries}")
print(f"清理后的退出信号\n{clean_exits}")

## clean_enex_nb
二维版本的 `clean_enex_1d_nb`。

### 源码

```python
@njit(cache=True)
def clean_enex_nb(entries: tp.Array2d,
                  exits: tp.Array2d,
                  entry_first: bool) -> tp.Tuple[tp.Array2d, tp.Array2d]:

    entries_out = np.empty(entries.shape, dtype=np.bool_)
    exits_out = np.empty(exits.shape, dtype=np.bool_)

    for col in range(entries.shape[1]):
        entries_out[:, col], exits_out[:, col] = clean_enex_1d_nb(entries[:, col], exits[:, col], entry_first)
    return entries_out, exits_out
```

# Random

## rand_choice_nb
随机选择函数，指定索引 `from_i` 到 `to_i` 内随机选择 `n/n[col]` 个索引。

参数
- `from_i`（int）: 选择范围的起始索引(包含)
- `to_i`（int）: 选择范围的结束索引(不包含)
- `col`（int）: 当前列索引，用于灵活索引
- `n`（int或array-like）: 要选择的信号数量
  - 标量：所有列使用相同数量
  - 数组：每列可以有不同数量(使用灵活索引)

返回：选中的信号位置索引数组，按升序排列

### 源码
```python
@njit(cache=True)
def rand_choice_nb(from_i: int, to_i: int, col: int, n: tp.MaybeArray[int]) -> tp.Array1d:

    ns = np.asarray(n)
    size = min(to_i - from_i, flex_select_auto_nb(ns, 0, col, True))
    return from_i + np.random.choice(to_i - from_i, size=size, replace=False)
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import rand_choice_nb

indices = rand_choice_nb(10, 20, 0, 3)
print(f"indices: {indices}")

# 使用数组指定不同列的选择数量
n_array = np.array([2, 3, 1])  # 第0列选2个，第1列选3个，第2列选1个
indices_col1 = rand_choice_nb(5, 15, 1, n_array)  # 为第1列选择3个
print(f"indices_col1: {indices_col1}")

## generate_rand_nb
创建一个形状为 `shape` 的 `bool` 矩阵，每列随机有 `n` 个位置为 `True`。

参数
- `shape`（tuple）：目标信号矩阵的形状，格式为(时间, 资产)
- `n`（int或array-like）：每列要生成的信号数量
  - 标量：所有列使用相同数量
  - 数组：支持每列不同数量(使用灵活索引)
- `seed` (int, optional)：随机种子，用于确保结果可重现
  - None：使用当前随机状态
  - 整数：设置特定随机种子


### 源码
```python
@njit
def generate_rand_nb(shape: tp.Shape, n: tp.MaybeArray[int], seed: tp.Optional[int] = None) -> tp.Array2d:

    if seed is not None:
        np.random.seed(seed)
    return generate_nb(
        shape,
        False,
        rand_choice_nb, n
    )
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import generate_rand_nb

# 为3个资产、100个时间点的矩阵，每列随机生成5个信号
signals = generate_rand_nb((100, 3), n=5, seed=42)
print(f"总信号数: {signals.sum()}")  # 应该是15个信号
print(f"信号矩阵: {signals}")

# 不同列生成不同数量的信号
n_per_col = np.array([3, 5, 2])
signals = generate_rand_nb((50, 3), n=n_per_col, seed=123)
print(f"总信号数: {signals.sum()}")  # 应该是10个信号
print(f"信号矩阵: {signals}")

## rand_by_prob_choice_nb
在索引范围 $[$`from_i,to_i`$)$，随机选择若干索引，返回这些索引组成的数组。 
- 确定每个索引对应的概率 $p_i$
  - `prob` 是标量：$p_i=$`prob`
  - `prob` 是一维数组：$p_i=$`prob[i]`
  - `prob` 是二维数组：$p_i=$`prob[i,col]`
- 每个索引随机生成 ${u_i} \sim U\left( {0,1} \right)$
- 如果 ${u_i} < {p_i}$，则在返回数组中添加该索引

### 源码
```python
@njit(cache=True)
def rand_by_prob_choice_nb(from_i: int,
                           to_i: int,
                           col: int,
                           prob: tp.MaybeArray[float],
                           pick_first: bool,
                           temp_idx_arr: tp.Array1d,
                           flex_2d: bool) -> tp.Array1d:

    probs = np.asarray(prob)
    j = 0
    for i in range(from_i, to_i):
        if np.random.uniform(0, 1) < flex_select_auto_nb(probs, i, col, flex_2d):  # [0, 1)
            temp_idx_arr[j] = i
            j += 1
            if pick_first:
                break
    return temp_idx_arr[:j]
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import rand_by_prob_choice_nb

# 固定概率0.1的信号生成
temp_arr = np.empty(50, dtype=np.int64)
indices = rand_by_prob_choice_nb(0, 50, 0, 0.1, False, temp_arr, False)
print(f"固定概率0.1的信号生成: {indices}")

# 时变概率：开盘时概率高，收盘时概率低
prob_array = np.linspace(0.2, 0.05, 100)  # 递减概率
indices = rand_by_prob_choice_nb(0, 100, 0, prob_array, False, temp_arr, True)
print(f"时变概率：{indices}")

## generate_rand_by_prob_nb
类似于函数 `rand_by_prob_choice_nb`，但是有如下区别：
- 返回的是形状为 `shape` 的 `bool` 矩阵，${u} < {p}$ 对应的位置置为 `True`
- 如果 `pick_first=True`，每列只保留第一个 `True`

### 源码
```python
@njit
def generate_rand_by_prob_nb(shape: tp.Shape,
                             prob: tp.MaybeArray[float],
                             pick_first: bool,
                             flex_2d: bool,
                             seed: tp.Optional[int] = None) -> tp.Array2d:

    if seed is not None:
        np.random.seed(seed)
    temp_idx_arr = np.empty((shape[0],), dtype=np.int64)
    return generate_nb(
        shape,
        pick_first,
        rand_by_prob_choice_nb, prob, pick_first, temp_idx_arr, flex_2d
    )
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import generate_rand_by_prob_nb

signals = generate_rand_by_prob_nb(
    (100, 3), prob=0.05, pick_first=False, flex_2d=False, seed=42
)
print(f"signals: {signals}")

# 2. 时变概率信号生成
# 交易时间概率高，非交易时间概率低
time_probs = np.where(
    (np.arange(100) % 24 >= 9) & (np.arange(100) % 24 <= 16),
    0.1, 0.02  # 交易时间10%概率，其他时间2%概率
)
signals = generate_rand_by_prob_nb(
    (100, 3), prob=time_probs, pick_first=False, flex_2d=True, seed=42
)

# 3. 个股特定概率矩阵
# 不同股票在不同时间有不同的信号概率
prob_matrix = np.random.beta(2, 8, (100, 3))  # Beta分布概率
signals = generate_rand_by_prob_nb(
    (100, 3), prob=prob_matrix, pick_first=False, flex_2d=True, seed=42
)

# Random exits

## generate_rand_ex_nb
函数 `generate_ex_nb` 的随机版本。出场信号是在 $from_k,from_k + 1, \cdots ,to_k - 1$ 中随机选择的。

### 源码
```python
@njit
def generate_rand_ex_nb(entries: tp.Array2d,
                        wait: int,
                        until_next: bool,
                        skip_until_exit: bool,
                        seed: tp.Optional[int] = None) -> tp.Array2d:

    if seed is not None:
        np.random.seed(seed)
    return generate_ex_nb(
        entries,
        wait,
        until_next,
        skip_until_exit,
        True,
        rand_choice_nb, 1
    )
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import generate_rand_ex_nb

entries = np.zeros((10, 2), dtype=bool)
entries[[1, 3, 6], 0] = True  # 第一个资产的入场信号
entries[[1, 4, 7], 1] = True  # 第二个资产的入场信号

# 生成随机退出信号
random_exits = generate_rand_ex_nb(
    entries, wait=1, until_next=True, 
    skip_until_exit=False, seed=42
)

print(f"入场信号\n: {entries}")
print(f"退出信号\n: {random_exits}")

## generate_rand_ex_by_prob_nb
函数 `generate_ex_nb` 的随机版本。出场信号是在 $[$`from_k,to_k`$)$ 中，类似于 `rand_by_prob_choice_nb` 中的伯努利实验得到的。

### 源码
```python
@njit
def generate_rand_ex_by_prob_nb(entries: tp.Array2d,
                                prob: tp.MaybeArray[float],
                                wait: int,
                                until_next: bool,
                                skip_until_exit: bool,
                                flex_2d: bool,
                                seed: tp.Optional[int] = None) -> tp.Array2d:

    if seed is not None:
        np.random.seed(seed)
    temp_idx_arr = np.empty((entries.shape[0],), dtype=np.int64)
    return generate_ex_nb(
        entries,
        wait,
        until_next,
        skip_until_exit,
        True,
        rand_by_prob_choice_nb, prob, True, temp_idx_arr, flex_2d
    )
```

## generate_rand_enex_nb
生成形状为 `shape` 的入场和出场 `bool` 矩阵。
- 对于每列 col，生成 `n/n[col]` 对交替的 $入场 \to 出场 \to 入场 \cdots  \to 入场 \to 出场$
- `entry_wait`：入场等待时间，从一个出场信号到下一个入场信号的最小时间间隔
- `exit_wait`：出场等待时间，从一个入场信号到对应出场信号的最小时间间隔

### 源码
```python
@njit
def generate_rand_enex_nb(shape: tp.Shape,
                          n: tp.MaybeArray[int],
                          entry_wait: int,
                          exit_wait: int,
                          seed: tp.Optional[int] = None) -> tp.Tuple[tp.Array2d, tp.Array2d]:

    if seed is not None:
        np.random.seed(seed)
    entries = np.full(shape, False)
    exits = np.full(shape, False)
    if entry_wait == 0 and exit_wait == 0:
        raise ValueError("entry_wait and exit_wait cannot be both 0")
    ns = np.asarray(n)

    if entry_wait == 1 and exit_wait == 1:
        # Basic case
        both = generate_rand_nb(shape, ns * 2, seed=None)
        for col in range(both.shape[1]):
            both_idxs = np.flatnonzero(both[:, col])
            entries[both_idxs[0::2], col] = True
            exits[both_idxs[1::2], col] = True
    else:
        for col in range(shape[1]):
            _n = flex_select_auto_nb(ns, 0, col, True)
            if _n == 1:
                entry_idx = np.random.randint(0, shape[0] - exit_wait)
                entries[entry_idx, col] = True
            else:
                # Minimum range between two entries
                min_range = entry_wait + exit_wait

                # Minimum total range between first and last entry
                min_total_range = min_range * (_n - 1)
                if shape[0] < min_total_range + exit_wait + 1:
                    raise ValueError("Cannot take a larger sample than population")

                # We should decide how much space should be allocate before first and after last entry
                # Maximum space outside of min_total_range
                max_free_space = shape[0] - min_total_range - 1

                # If min_total_range is tiny compared to max_free_space, limit it
                # otherwise we would have huge space before first and after last entry
                # Limit it such as distribution of entries mimics uniform
                free_space = min(max_free_space, 3 * shape[0] // (_n + 1))

                # What about last exit? it requires exit_wait space
                free_space -= exit_wait

                # Now we need to distribute free space among three ranges:
                # 1) before first, 2) between first and last added to min_total_range, 3) after last
                # We do 2) such that min_total_range can freely expand to maximum
                # We allocate twice as much for 3) as for 1) because an exit is missing
                rand_floats = uniform_summing_to_one_nb(6)
                chosen_spaces = rescale_float_to_int_nb(rand_floats, (0, free_space), free_space)
                first_idx = chosen_spaces[0]
                last_idx = shape[0] - np.sum(chosen_spaces[-2:]) - exit_wait - 1

                # Selected range between first and last entry
                total_range = last_idx - first_idx

                # Maximum range between two entries within total_range
                max_range = total_range - (_n - 2) * min_range

                # Select random ranges within total_range
                rand_floats = uniform_summing_to_one_nb(_n - 1)
                chosen_ranges = rescale_float_to_int_nb(rand_floats, (min_range, max_range), total_range)

                # Translate them into entries
                entry_idxs = np.empty(_n, dtype=np.int64)
                entry_idxs[0] = first_idx
                entry_idxs[1:] = chosen_ranges
                entry_idxs = np.cumsum(entry_idxs)
                entries[entry_idxs, col] = True

        # Generate exits
        for col in range(shape[1]):
            entry_idxs = np.flatnonzero(entries[:, col])
            for j in range(len(entry_idxs)):
                entry_i = entry_idxs[j] + exit_wait
                if j < len(entry_idxs) - 1:
                    exit_i = entry_idxs[j + 1] - entry_wait
                else:
                    exit_i = entries.shape[0] - 1
                i = np.random.randint(exit_i - entry_i + 1)
                exits[entry_i + i, col] = True
    return entries, exits
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import generate_rand_enex_nb

# 1. 基础随机信号对生成
entries, exits = generate_rand_enex_nb(
    shape=(20, 3), n=5, entry_wait=1, exit_wait=1, seed=42
)
print(f"入场信号\n: {entries}")  # 应该是15
print(f"退出信号\n: {exits}")    # 应该是15

# 2. 不同列生成不同数量的信号对
n_per_col = np.array([3, 5, 2])  # 不同列的信号对数量
entries, exits = generate_rand_enex_nb(
    shape=(200, 3), n=n_per_col, entry_wait=5, exit_wait=3, seed=123
)

# 验证每列的信号对数量
for col in range(3):
    entry_count = entries[:, col].sum()
    exit_count = exits[:, col].sum()
    print(f"列{col}: 入场={entry_count}, 退出={exit_count}")

## rand_enex_apply_nb

### 源码
```python
def rand_enex_apply_nb(input_shape: tp.Shape,
                       n: tp.MaybeArray[int],
                       entry_wait: int,
                       exit_wait: int) -> tp.Tuple[tp.Array2d, tp.Array2d]:

    return generate_rand_enex_nb(input_shape, n, entry_wait, exit_wait)
```

## generate_rand_enex_by_prob_nb
生成入场信号和退出信号。

流程：遍历每一列 $n$（时间×资产，$T \times N$）
- `prev_i` 和 `prev_prev_i` 分别记录上一个、上上一个 `True` 索引
- 处理顺序：入场——>出场——>入场——>出场——>...
  - 处理入场时
    - `from_i = prev_i + entry_wait`，第一次处理时，`from_i=0`
    - `to_i =` $T$
    - 在索引范围 $[$`from_i,to_i`$)$，随机选择若干索引，返回这些索引组成的数组 `idxs`
      - 确定每个索引对应的概率 $p_i$
        - `prob` 是标量：$p_i=$`prob`
        - `prob` 是一维数组：$p_i=$`prob[i]`
      - 每个索引随机生成 ${u_i} \sim U\left( {0,1} \right)$
      - 如果 ${u_i} < {p_i}$，则在返回数组中添加该索引

  - 处理出场时
    - `from_i = prev_i + exit_wait`
    - `to_i =` $T$
    - 与处理入场时类似，返回随机生成的索引数组 `idxs`
  - 根据 `entry_pick_first/exit_pick_first=True/False`，置 `entries/exits[idxs[0]/idxs, col]=True`
  - 每次处理完一次入/出场，根据 `entry_pick_first/exit_pick_first`
    - `True`
      - `entries/exits[idxs[0], col]=True`
      - `prev_prev_i = prev_i`，`prev_i = idxs[0]`
    - `False`
      - `entries/exits[idxs, col]=True`
      - `prev_prev_i = prev_i`，`prev_i = idxs[-1]`

### 源码
```python
@njit
def generate_rand_enex_by_prob_nb(shape: tp.Shape,
                                  entry_prob: tp.MaybeArray[float],
                                  exit_prob: tp.MaybeArray[float],
                                  entry_wait: int,
                                  exit_wait: int,
                                  entry_pick_first: bool,
                                  exit_pick_first: bool,
                                  flex_2d: bool,
                                  seed: tp.Optional[int] = None) -> tp.Tuple[tp.Array2d, tp.Array2d]:

    if seed is not None:
        np.random.seed(seed)
    temp_idx_arr = np.empty((shape[0],), dtype=np.int64)
    return generate_enex_nb(
        shape,
        entry_wait,
        exit_wait,
        entry_pick_first,
        exit_pick_first,
        rand_by_prob_choice_nb, (entry_prob, entry_pick_first, temp_idx_arr, flex_2d),
        rand_by_prob_choice_nb, (exit_prob, exit_pick_first, temp_idx_arr, flex_2d)
    )
```

# Stop exits

## first_choice_nb
返回数组 `a` 中列 `col` 行范围 $[$`from_i,to_i`$)$ 中第一个 True 信号的索引。

### 源码
```python
@njit(cache=True)
def first_choice_nb(from_i: int, to_i: int, col: int, a: tp.Array2d) -> tp.Array1d:
    out = np.empty((1,), dtype=np.int64)
    for i in range(from_i, to_i):
        if a[i, col]:
            out[0] = i
            return out
    return out[:0]  # empty
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import first_choice_nb

signals = np.array([[False, True], [True, False], [False, True]])

# 在第0列的[0,3)范围内寻找第一个信号
first_idx = first_choice_nb(0, 3, 0, signals)
print(f"first_idx: {first_idx}")

# 在第1列寻找
first_idx = first_choice_nb(0, 3, 1, signals)
print(f"first_idx: {first_idx}")

## stop_choice_nb
寻找价格序列 `ts[[from_i, to_i), col]` 中的 {固定/移动，止盈/止损} 信号索引。

具体流程：
- 考虑入场价格 `entry_price = ts[from_i-wait, col]`
- `init_trailing = trailing[from_i-wait, col]` 决定是移动/固定
- `init_stop = stop[from_i-wait, col]` 的正负情况决定止盈/止损
- 遍历价格矩阵 `ts` 的 $[$`from_i,to_i`$)$ 行，`col` 列的每一个行索引 `i`
  - 固定止盈：如果 `ts[i, col] >= entry_price * (1 + init_stop)`
  - 固定止损：如果 `ts[i, col] <= entry_price * (1 + init_stop)`
  - 移动止盈：如果 `ts[i, col] >= min_low * (1 + init_stop)`
    - 其中 `min_low` 是 $[$`from_i,i`$)$ 这一段的最低价
  - 移动止损：如果 `ts[i, col] <= max_high * (1 + init_stop)`
    - 其中 `max_high` 是 $[$`from_i,i`$)$ 这一段的最高价

    如果符合上述条件，就将 `i` 记录到 `temp_idx_arr`
- 如果 `pick_first = True`，找到第一个符合条件的索引后就立即返回


### 源码
```python
@njit(cache=True)
def stop_choice_nb(from_i: int,
                   to_i: int,
                   col: int,
                   ts: tp.ArrayLike,
                   stop: tp.MaybeArray[float],
                   trailing: tp.MaybeArray[bool],
                   wait: int,
                   pick_first: bool,
                   temp_idx_arr: tp.Array1d,
                   flex_2d: bool) -> tp.Array1d:

    j = 0
    init_i = from_i - wait
    init_ts = flex_select_auto_nb(ts, init_i, col, flex_2d)
    init_stop = flex_select_auto_nb(np.asarray(stop), init_i, col, flex_2d)
    init_trailing = flex_select_auto_nb(np.asarray(trailing), init_i, col, flex_2d)
    max_high = min_low = init_ts

    for i in range(from_i, to_i):
        if not np.isnan(init_stop):
            if init_trailing:
                if init_stop >= 0:
                    # Trailing stop buy
                    curr_stop_price = min_low * (1 + abs(init_stop))
                else:
                    # Trailing stop sell
                    curr_stop_price = max_high * (1 - abs(init_stop))
            else:
                curr_stop_price = init_ts * (1 + init_stop)

        # Check if stop price is within bar
        curr_ts = flex_select_auto_nb(ts, i, col, flex_2d)
        if not np.isnan(init_stop):
            if init_stop >= 0:
                exit_signal = curr_ts >= curr_stop_price
            else:
                exit_signal = curr_ts <= curr_stop_price
            if exit_signal:
                temp_idx_arr[j] = i
                j += 1
                if pick_first:
                    return temp_idx_arr[:1]

        # Keep track of lowest low and highest high if trailing
        if init_trailing:
            if curr_ts < min_low:
                min_low = curr_ts
            elif curr_ts > max_high:
                max_high = curr_ts
    return temp_idx_arr[:j]
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import stop_choice_nb

# 创建价格时间序列
prices = np.array([[100, 105, 98, 110, 95]]).T  # 5个时间点的价格
temp_arr = np.empty(5, dtype=np.int64)
print(f"prices: {prices}")

# 固定10%止损
stop_indices = stop_choice_nb(
    0, 5, 0, prices, stop=-0.1, trailing=False,
    wait=0, pick_first=True, temp_idx_arr=temp_arr, flex_2d=True
)
print(f"固定10%止损索引: {stop_indices}")
# 移动10%止损
trail_indices = stop_choice_nb(
    0, 5, 0, prices, stop=-0.1, trailing=True,
    wait=0, pick_first=True, temp_idx_arr=temp_arr, flex_2d=True
)
print(f"移动10%止损索引: {trail_indices}")

## generate_stop_ex_nb
为每个入场信号生成对应的退出信号。
- 将 `stop_choice_nb` 作为索引选择函数，`(ts, stop, trailing, wait, pick_first)` 是其参数
- 将索引选择函数传递给 `generate_ex_nb` 来生成

完整过程：

（1）遍历每一列，找到每一列的所有入场信号
- 对于 `entries` $E \in {\left\{ {bool} \right\}^{T \times N}}$ 的每一列 $n$（每个资产）
- 找到所有入场信号的索引集合 ${1_n} = \left\{ {{t_1},{t_2}, \cdots ,{t_K}} \right\}$，其中 ${E_{{t_{k,n}}}} = true$

（2）对于每个入场信号，确定其出场信号的搜索区间
- 对于每个入场信号 $t_k$
  - 起点为 $fro{m_k} = {t_k} + wait$
  - 终点：如果 `until_next = True` 并且存在下一个入场信号 $t_{k+1}$，则 $to_k=t_{k+1}$；否则 $to_k=T$

（3）确定出场信号
- 令 $O_k$ = `stop_choice_nb`$\left( {fro{m_k},t{o_k},n} \right)$ 即寻找价格序列 `ts[[from_k, to_k), col]` 中的 {固定/移动，止盈/止损} 信号索引
  - 考虑入场价格 `entry_price = ts[from_k-wait, col]`
  - `init_trailing = trailing[from_k-wait, col]` 决定是移动/固定
  - `init_stop = stop[from_k-wait, col]` 的正负情况决定止盈/止损
  - 遍历价格矩阵 `ts` 的 $[$`from_k,to_k`$)$ 行，`col` 列的每一个行索引 `i`
    - 固定止盈：如果 `ts[i, col] >= entry_price * (1 + init_stop)`
    - 固定止损：如果 `ts[i, col] <= entry_price * (1 + init_stop)`
    - 移动止盈：如果 `ts[i, col] >= min_low * (1 + init_stop)`
      - 其中 `min_low` 是 $[$`from_k,i`$)$ 这一段的最低价
    - 移动止损：如果 `ts[i, col] <= max_high * (1 + init_stop)`
      - 其中 `max_high` 是 $[$`from_k,i`$)$ 这一段的最高价

    如果符合上述条件，就将 `i` 记录到 `temp_idx_arr`
- 如果 `pick_first = True`，找到第一个符合条件的索引后就立即返回
- 如果 `pick_first = True`，只取 $O_k$ 的第一个索引 ${O_{k,1}}$，令 ${X_{{O_{k,1}},n}} = True$；否则 ${X_{{O_k},n}} = True$

注意：如果 `skip_until_exit=True`，那么下一个入场信号必须在上一个出场信号之后才会被处理。

### 源码
```python
@njit
def generate_stop_ex_nb(entries: tp.Array2d,
                        ts: tp.ArrayLike,
                        stop: tp.MaybeArray[float],
                        trailing: tp.MaybeArray[bool],
                        wait: int,
                        until_next: bool,
                        skip_until_exit: bool,
                        pick_first: bool,
                        flex_2d: bool) -> tp.Array2d:

    temp_idx_arr = np.empty((entries.shape[0],), dtype=np.int64)
    return generate_ex_nb(
        entries,
        wait,
        until_next,
        skip_until_exit,
        pick_first,
        stop_choice_nb,
        ts,
        stop,
        trailing,
        wait,
        pick_first,
        temp_idx_arr,
        flex_2d
    )
```

## generate_stop_enex_nb
基于止损逻辑生成入场和出场信号。

参数：
- `entries` (np.array): 初始入场信号模板，用于信号激活
- `ts` (array-like): 价格时间序列，用于止损计算
- `stop` (float或array-like): 止损阈值设置，参见stop_choice_nb
- `trailing` (bool或array-like): 移动止损配置，参见stop_choice_nb
- `entry_wait` (int): 入场信号间的最小等待周期
- `exit_wait` (int): 退出信号间的最小等待周期
- `pick_first` (bool): 是否只选择第一个触发的止损信号
- `flex_2d` (bool): 灵活索引的二维模式标识

返回：tuple[np.array, np.array]: (新的入场信号矩阵, 止损退出信号矩阵)
- 两个矩阵形状都与entries相同
- 新入场信号是原始信号的清理版本
- 退出信号严格对应每个有效的入场信号

完整流程：遍历每一列 $n$（时间×资产，$T \times N$）
- `prev_i` 和 `prev_prev_i` 分别记录上一个、上上一个 `True` 索引
- 处理顺序：入场——>出场——>入场——>出场——>...
  - 处理入场时
    - `from_i = prev_i + entry_wait`，第一次处理时，`from_i=0`
    - `to_i =` $T$
    - 调用 `idxs = first_choice_nb(from_i, to_i, col, *entry_args)`
      - 即返回 `entries` 中列 `col` 行范围 $[$`from_i,to_i`$)$ 中第一个 True 信号的索引
  - 处理出场时
    - `from_i = prev_i + exit_wait`
    - `to_i =` $T$
    - 调用 `idxs = stop_choice_nb(from_i, to_i, col, *exit_args)`，即寻找价格序列 `ts[[from_i, to_i), col]` 中的 {固定/移动，止盈/止损} 信号索引。
      - 考虑入场价格 `entry_price = ts[from_i-wait, col]`
      - `init_trailing = trailing[from_i-wait, col]` 决定是移动/固定
      - `init_stop = stop[from_i-wait, col]` 的正负情况决定止盈/止损
      - 遍历价格矩阵 `ts` 的 $[$`from_i,to_i`$)$ 行，`col` 列的每一个行索引 `i`
        - 固定止盈：如果 `ts[i, col] >= entry_price * (1 + init_stop)`
        - 固定止损：如果 `ts[i, col] <= entry_price * (1 + init_stop)`
        - 移动止盈：如果 `ts[i, col] >= min_low * (1 + init_stop)`
          - 其中 `min_low` 是 $[$`from_i,i`$)$ 这一段的最低价
        - 移动止损：如果 `ts[i, col] <= max_high * (1 + init_stop)`
          - 其中 `max_high` 是 $[$`from_i,i`$)$ 这一段的最高价

        如果符合上述条件，就将 `i` 记录到 `temp_idx_arr`
- 如果 `pick_first = True`，找到第一个符合条件的索引后就立即返回
  - 根据 `entry_pick_first/exit_pick_first=True/False`，置 `entries/exits[idxs[0]/idxs, col]=True`
  - 每次处理完一次入/出场，根据 `entry_pick_first/exit_pick_first`
    - `True`
      - `entries/exits[idxs[0], col]=True`
      - `prev_prev_i = prev_i`，`prev_i = idxs[0]`
    - `False`
      - `entries/exits[idxs, col]=True`
      - `prev_prev_i = prev_i`，`prev_i = idxs[-1]`

### 源码
```python
@njit
def generate_stop_enex_nb(entries: tp.Array2d,
                          ts: tp.Array,
                          stop: tp.MaybeArray[float],
                          trailing: tp.MaybeArray[bool],
                          entry_wait: int,
                          exit_wait: int,
                          pick_first: bool,
                          flex_2d: bool) -> tp.Tuple[tp.Array2d, tp.Array2d]:

    temp_idx_arr = np.empty((entries.shape[0],), dtype=np.int64)
    return generate_enex_nb(
        entries.shape,
        entry_wait,
        exit_wait,
        True,
        pick_first,
        first_choice_nb, (entries,),
        stop_choice_nb, (ts, stop, trailing, exit_wait, pick_first, temp_idx_arr, flex_2d)
    )
```

### 例子

In [ ]:
import numpy as np
import vectorbt as vbt
from vectorbt.signals.nb import generate_stop_enex_nb

prices = np.random.randn(100, 3).cumsum(axis=0) + 100

ma_fast = vbt.MA.run(prices, window=5).ma
ma_slow = vbt.MA.run(prices, window=20).ma

raw_entries = (ma_fast.values > ma_slow.values) & (np.roll(ma_fast.values, 1, axis=0) <= np.roll(ma_slow.values, 1, axis=0))
# 注意：np.roll 会将第0行移到最后一行，首行会用最后一行的值，通常首行应设为False
raw_entries[0, :] = False

# 生成带止损的完整交易信号
clean_entries, stop_exits = generate_stop_enex_nb(
    raw_entries, prices,
    stop=-0.05,          # 5%固定止损
    trailing=False,      # 使用固定止损
    entry_wait=2,        # 入场信号间隔2个周期
    exit_wait=1,         # 止损后1个周期才能重新入场
    pick_first=True,     # 只选择第一个止损触发点
    flex_2d=True
)

print(f"原始入场信号数: {raw_entries.sum()}")
print(f"清理后入场信号数: {clean_entries.sum()}")
print(f"止损退出信号数: {stop_exits.sum()}")

## ohlc_stop_choice_nb

### 多头和空头
**多头**是指投资者预期资产价格会上涨，因此买入资产持有，等待价格上涨后卖出获利的交易策略。
- 买入 → 持有 → 价格上涨 → 卖出 → 获利

**空头**是指投资者预期资产价格会下跌，因此先借入资产卖出，等待价格下跌后低价买回归还，从而获利的交易策略。
- 借入 → 卖出 → 价格下跌 → 买入 → 归还 → 获利

注意：多头和空头的止损和止盈逻辑是相反的。

### 函数逻辑

寻找 OHLC 数据 `[from_i, to_i), col` 中的止盈/止损信号索引。

具体流程：
- 初始开盘价 `init_open = open[from_i-wait, col]`
- 止损阈值 `init_sl_stop = sl_stop[from_i-wait, col]`
- 止盈阈值 `init_tp_stop = tp_stop[from_i-wait, col]`
- 是否启用*移动* `init_sl_trail = sl_trail[from_i-wait, col]`
- 空头还是多头 `init_reverse = reverse[from_i-wait, col]`
- 是否可以开盘中交易 `is_open_safe`

过程：遍历开盘价/最高价/最低价/收盘价 `open/high/low/close` 的 $[$`from_i,to_i`$)$ 行，`col` 列的每一个行索引 `i`
- 计算止损价 `curr_sl_stop_price`
  - 移动、空头：`min_p * (1 + init_sl_stop)`
    - `min_p` 是 $[$`from_k,i`$)$ 这一段的最低价 `low` 的最小值
  - 移动、多头：`max_p * (1 - init_sl_stop)`
    - `max_p` 是 $[$`from_k,i`$)$ 这一段的最高价 `high` 的最大值
  - 固定、空头：`init_open * (1 + init_sl_stop)`
  - 固定、多头：`init_open * (1 - init_sl_stop)`
- 计算止盈价 `curr_tp_stop_price`
  - 空头：`init_open * (1 - init_tp_stop)`
  - 多头：`init_open * (1 + init_tp_stop)`
- 确定当前可使用的价格 `curr_high` 和 `curr_low`
  - 开盘中可交易：`curr_high=high[i,col], curr_low=low[i,col]`
  - 开盘中不可交易：`curr_high = curr_low = close[i,col]`
- 检查是否触发止损：多头且`curr_low`<=止损价，或者空头且`curr_high`>=止损价
  - 记录触发价格 `stop_price_out[i, col] = curr_sl_stop_price`
  - 根据*移动*或*固定*，记录止损类型 `stop_type_out[i, col] = StopType.TrailStop/StopType.StopLoss`
- 如果没有触发止损则检查是否触发止盈：多头且`curr_high`>=止盈价，或者空头且`curr_low`<=止盈价
  - 记录触发价格 `stop_price_out[i, col] = curr_tp_stop_price`
  - 记录止盈类型 `stop_type_out[i, col] = StopType.TakeProfit`
- 如果止损/止盈触发，将 `i` 记录到 `temp_idx_arr`
  - 如果 `pick_first = True`，找到第一个符合条件的索引后就立即返回

### 源码
```python
@njit(cache=True)
def ohlc_stop_choice_nb(from_i: int,
                        to_i: int,
                        col: int,
                        open: tp.ArrayLike,
                        high: tp.ArrayLike,
                        low: tp.ArrayLike,
                        close: tp.ArrayLike,
                        stop_price_out: tp.Array2d,
                        stop_type_out: tp.Array2d,
                        sl_stop: tp.MaybeArray[float],
                        sl_trail: tp.MaybeArray[bool],
                        tp_stop: tp.MaybeArray[float],
                        reverse: tp.MaybeArray[bool],
                        is_open_safe: bool,
                        wait: int,
                        pick_first: bool,
                        temp_idx_arr: tp.Array1d,
                        flex_2d: bool) -> tp.Array1d:

    init_i = from_i - wait
    init_open = flex_select_auto_nb(open, init_i, col, flex_2d)
    init_sl_stop = flex_select_auto_nb(np.asarray(sl_stop), init_i, col, flex_2d)
    if init_sl_stop < 0:
        raise ValueError("Stop value must be 0 or greater")
    init_sl_trail = flex_select_auto_nb(np.asarray(sl_trail), init_i, col, flex_2d)
    init_tp_stop = flex_select_auto_nb(np.asarray(tp_stop), init_i, col, flex_2d)
    if init_tp_stop < 0:
        raise ValueError("Stop value must be 0 or greater")
    init_reverse = flex_select_auto_nb(np.asarray(reverse), init_i, col, flex_2d)
    max_p = min_p = init_open
    j = 0

    for i in range(from_i, to_i):
        # Resolve current bar
        _open = flex_select_auto_nb(open, i, col, flex_2d)
        _high = flex_select_auto_nb(high, i, col, flex_2d)
        _low = flex_select_auto_nb(low, i, col, flex_2d)
        _close = flex_select_auto_nb(close, i, col, flex_2d)
        if np.isnan(_open):
            _open = _close
        if np.isnan(_low):
            _low = min(_open, _close)
        if np.isnan(_high):
            _high = max(_open, _close)

        # Calculate stop price
        if not np.isnan(init_sl_stop):
            if init_sl_trail:
                if init_reverse:
                    curr_sl_stop_price = min_p * (1 + init_sl_stop)
                else:
                    curr_sl_stop_price = max_p * (1 - init_sl_stop)
            else:
                if init_reverse:
                    curr_sl_stop_price = init_open * (1 + init_sl_stop)
                else:
                    curr_sl_stop_price = init_open * (1 - init_sl_stop)
        if not np.isnan(init_tp_stop):
            if init_reverse:
                curr_tp_stop_price = init_open * (1 - init_tp_stop)
            else:
                curr_tp_stop_price = init_open * (1 + init_tp_stop)

        # Check if stop price is within bar
        if i > init_i or is_open_safe:
            # is_open_safe means open is either open or any other price before it
            # so it's safe to use high/low at entry bar
            curr_high = _high
            curr_low = _low
        else:
            # Otherwise, we can only use close price at entry bar
            curr_high = curr_low = _close

        exit_signal = False
        if not np.isnan(init_sl_stop):
            if (not init_reverse and curr_low <= curr_sl_stop_price) or \
                    (init_reverse and curr_high >= curr_sl_stop_price):
                exit_signal = True
                stop_price_out[i, col] = curr_sl_stop_price
                if init_sl_trail:
                    stop_type_out[i, col] = StopType.TrailStop
                else:
                    stop_type_out[i, col] = StopType.StopLoss
        if not exit_signal and not np.isnan(init_tp_stop):
            if (not init_reverse and curr_high >= curr_tp_stop_price) or \
                    (init_reverse and curr_low <= curr_tp_stop_price):
                exit_signal = True
                stop_price_out[i, col] = curr_tp_stop_price
                stop_type_out[i, col] = StopType.TakeProfit
        if exit_signal:
            temp_idx_arr[j] = i
            j += 1
            if pick_first:
                return temp_idx_arr[:1]

        # Keep track of highest high if trailing
        if init_sl_trail:
            if curr_low < min_p:
                min_p = curr_low
            if curr_high > max_p:
                max_p = curr_high

    return temp_idx_arr[:j]
```

### 例子

In [ ]:
import numpy as np
import pandas as pd
from vectorbt.signals.nb import ohlc_stop_choice_nb

# 创建OHLC数据
ohlc_data = pd.DataFrame({
    'open': [100, 102, 104, 103, 101],
    'high': [101, 105, 106, 104, 102], 
    'low': [99, 101, 103, 100, 99],
    'close': [102, 104, 103, 101, 100]
})

# 输出数组
stop_price_out = np.full((5, 1), np.nan)
stop_type_out = np.full((5, 1), -1, dtype=np.int64)
temp_arr = np.empty(5, dtype=np.int64)

# 5%移动止损 + 10%止盈
triggers = ohlc_stop_choice_nb(
    0, 5, 0,
    ohlc_data.open.values, ohlc_data.high.values,
    ohlc_data.low.values, ohlc_data.close.values,
    stop_price_out, stop_type_out,
    sl_stop=0.05, sl_trail=True, tp_stop=0.10, reverse=False,
    is_open_safe=True, wait=0, pick_first=True,
    temp_idx_arr=temp_arr, flex_2d=False
)

print(f"stop_price_out: {stop_price_out}")
print(f"stop_type_out: {stop_type_out}")
print(f"temp_arr: {temp_arr}")
print(f"triggers: {triggers}")

## generate_ohlc_stop_ex_nb
为每个入场信号生成对应的退出信号。
- 将 `ohlc_stop_choice_nb` 作为索引选择函数，将索引选择函数传递给 `generate_ex_nb` 来生成

完整过程：

（1）遍历每一列，找到每一列的所有入场信号
- 对于 `entries` $E \in {\left\{ {bool} \right\}^{T \times N}}$ 的每一列 $n$（每个资产）
- 找到所有入场信号的索引集合 ${1_n} = \left\{ {{t_1},{t_2}, \cdots ,{t_K}} \right\}$，其中 ${E_{{t_{k,n}}}} = true$

（2）对于每个入场信号，确定其出场信号的搜索区间
- 对于每个入场信号 $t_k$
  - 起点为 $fro{m_k} = {t_k} + wait$
  - 终点：如果 `until_next = True` 并且存在下一个入场信号 $t_{k+1}$，则 $to_k=t_{k+1}$；否则 $to_k=T$

（3）确定出场信号
- 令 $O_k$ = `ohlc_stop_choice_nb`$\left( {fro{m_k},t{o_k},n} \right)$
- 如果 `pick_first = True`，只取 $O_k$ 的第一个索引 ${O_{k,1}}$，令 ${X_{{O_{k,1}},n}} = True$；否则 ${X_{{O_k},n}} = True$

注意：如果 `skip_until_exit=True`，那么下一个入场信号必须在上一个出场信号之后才会被处理。

### 源码
```python
@njit
def generate_ohlc_stop_ex_nb(entries: tp.Array2d,
                             open: tp.ArrayLike,
                             high: tp.ArrayLike,
                             low: tp.ArrayLike,
                             close: tp.ArrayLike,
                             stop_price_out: tp.Array2d,
                             stop_type_out: tp.Array2d,
                             sl_stop: tp.MaybeArray[float],
                             sl_trail: tp.MaybeArray[bool],
                             tp_stop: tp.MaybeArray[float],
                             reverse: tp.MaybeArray[bool],
                             is_open_safe: bool,
                             wait: int,
                             until_next: bool,
                             skip_until_exit: bool,
                             pick_first: bool,
                             flex_2d: bool) -> tp.Array2d:

    temp_idx_arr = np.empty((entries.shape[0],), dtype=np.int64)
    return generate_ex_nb(
        entries,
        wait,
        until_next,
        skip_until_exit,
        pick_first,
        ohlc_stop_choice_nb,
        open,
        high,
        low,
        close,
        stop_price_out,
        stop_type_out,
        sl_stop,
        sl_trail,
        tp_stop,
        reverse,
        is_open_safe,
        wait,
        pick_first,
        temp_idx_arr,
        flex_2d
    )
```

## generate_ohlc_stop_enex_nb
基于止损逻辑生成入场和出场信号。

参数
- `entries`(np.array)：初始入场信号模板，用于信号激活
- `open/high/low/close`(array-like)：完整的OHLC价格数据时间序列
- `stop_price_out`(np.array)：输出数组，记录实际触发价格
- `stop_type_out`(np.array)：输出数组，记录触发类型
- `sl_stop`(float或array-like)：止损阈值设置，参见ohlc_stop_choice_nb
- `sl_trail`(bool或array-like)：移动止损配置，参见ohlc_stop_choice_nb
- `tp_stop`(float或array-like)：止盈阈值设置，参见ohlc_stop_choice_nb
- `reverse`(bool或array-like)：交易方向配置，参见ohlc_stop_choice_nb
- `is_open_safe`(bool)：开盘价安全性标识，参见ohlc_stop_choice_nb
- `entry_wait`(int)：入场信号间的最小等待周期
- `exit_wait`(int)：退出信号间的最小等待周期
- `pick_first`(bool)：是否只选择第一个触发的止损/止盈信号
- `flex_2d`(bool)：灵活索引的二维模式标识

返回值：tuple[np.array, np.array]: (清理后的入场信号矩阵, OHLC止损止盈退出信号矩阵)
- 两个矩阵形状都与entries相同
- 新入场信号是原始信号的清理和重组版本
- 退出信号严格对应每个有效入场信号的OHLC止损/止盈

完整流程：遍历每一列 $n$（时间×资产，$T \times N$）
- `prev_i` 和 `prev_prev_i` 分别记录上一个、上上一个 `True` 索引
- 处理顺序：入场——>出场——>入场——>出场——>...
  - 处理入场时
    - `from_i = prev_i + entry_wait`，第一次处理时，`from_i=0`
    - `to_i =` $T$
    - 调用 `idxs = first_choice_nb(from_i, to_i, col, *entry_args)`
      - 即返回 `entries` 中列 `col` 行范围 $[$`from_i,to_i`$)$ 中第一个 True 信号的索引
  - 处理出场时
    - `from_i = prev_i + exit_wait`
    - `to_i =` $T$
    - 调用 `idxs = ohlc_stop_choice_nb(from_i, to_i, col, *exit_args)`
      - 即寻找 OHLC 数据 `[from_i, to_i), col` 中的止盈/止损信号索引
  - 根据 `entry_pick_first/exit_pick_first=True/False`，置 `entries/exits[idxs[0]/idxs, col]=True`
  - 每次处理完一次入/出场，根据 `entry_pick_first/exit_pick_first`
    - `True`
      - `entries/exits[idxs[0], col]=True`
      - `prev_prev_i = prev_i`，`prev_i = idxs[0]`
    - `False`
      - `entries/exits[idxs, col]=True`
      - `prev_prev_i = prev_i`，`prev_i = idxs[-1]`

### 源码
```python
@njit
def generate_ohlc_stop_enex_nb(entries: tp.Array2d,
                               open: tp.ArrayLike,
                               high: tp.ArrayLike,
                               low: tp.ArrayLike,
                               close: tp.ArrayLike,
                               stop_price_out: tp.Array2d,
                               stop_type_out: tp.Array2d,
                               sl_stop: tp.MaybeArray[float],
                               sl_trail: tp.MaybeArray[bool],
                               tp_stop: tp.MaybeArray[float],
                               reverse: tp.MaybeArray[bool],
                               is_open_safe: bool,
                               entry_wait: int,
                               exit_wait: int,
                               pick_first: bool,
                               flex_2d: bool) -> tp.Tuple[tp.Array2d, tp.Array2d]:

    temp_idx_arr = np.empty((entries.shape[0],), dtype=np.int64)
    return generate_enex_nb(
        entries.shape,
        entry_wait,
        exit_wait,
        True,
        pick_first,
        first_choice_nb, (entries,),
        ohlc_stop_choice_nb, (
            open,
            high,
            low,
            close,
            stop_price_out,
            stop_type_out,
            sl_stop,
            sl_trail,
            tp_stop,
            reverse,
            is_open_safe,
            exit_wait,
            pick_first,
            temp_idx_arr,
            flex_2d
        )
    )
```

# Map and reduce ranges

## between_ranges_nb
记录布尔矩阵 `a` 中的每一对 `True`：
- 遍历 `a` 的每一列，找到每对 `True`，记录它们的
  - `id`：被找到的顺序
  - `col`：所在列
  - `start_idx`：起点 `True` 所在行
  - `end_idx`：终点 `True` 所在行
  - `status`：`RangeStatus.Closed`

### 源码
```python
@njit(cache=True)
def between_ranges_nb(a: tp.Array2d) -> tp.RecordArray:

    range_records = np.empty(a.shape[0] * a.shape[1], dtype=range_dt)
    ridx = 0

    for col in range(a.shape[1]):
        a_idxs = np.flatnonzero(a[:, col])
        if a_idxs.shape[0] > 1:
            for j in range(1, a_idxs.shape[0]):
                from_i = a_idxs[j - 1]
                to_i = a_idxs[j]
                range_records[ridx]['id'] = ridx
                range_records[ridx]['col'] = col
                range_records[ridx]['start_idx'] = from_i
                range_records[ridx]['end_idx'] = to_i
                range_records[ridx]['status'] = RangeStatus.Closed
                ridx += 1
    return range_records[:ridx]
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import between_ranges_nb

# 创建测试信号：两个资产的信号序列
signals = np.array([
    [True, False, True, False, True],    # 资产1: 位置0,2,4有信号
    [False, True, False, True, False]    # 资产2: 位置1,3有信号
]).T

# 分析信号间的范围
ranges = between_ranges_nb(signals)
print(ranges)

## between_two_ranges_nb
布尔矩阵 `a` 和 `b` 配对：
- 正向配对 `from_other=True`：遍历 `a` 的每一列，找到所有 `True` 位置，在 `b` 的对应行之后找到最接近的行索引
- 反向配对 `from_other=False`：遍历 `b` 的每一列，找到所有 `True` 位置，在 `a` 的对应行之前找到最接近的行索引
- 记录
  - `id`：被找到的顺序
  - `col`：所在列
  - `start_idx`：`a` 的 `True` 所在行
  - `end_idx`：`b` 的 `True` 所在行
  - `status`：`RangeStatus.Closed`


### 源码
```python
@njit(cache=True)
def between_two_ranges_nb(a: tp.Array2d, b: tp.Array2d, from_other: bool = False) -> tp.RecordArray:

    range_records = np.empty(a.shape[0] * a.shape[1], dtype=range_dt)
    ridx = 0

    for col in range(a.shape[1]):
        a_idxs = np.flatnonzero(a[:, col])
        if a_idxs.shape[0] > 0:
            b_idxs = np.flatnonzero(b[:, col])
            if b_idxs.shape[0] > 0:
                if from_other:
                    for j, to_i in enumerate(b_idxs):
                        valid_a_idxs = a_idxs[a_idxs <= to_i]
                        if len(valid_a_idxs) > 0:
                            from_i = valid_a_idxs[-1]  # preceding in a
                            range_records[ridx]['id'] = ridx
                            range_records[ridx]['col'] = col
                            range_records[ridx]['start_idx'] = from_i
                            range_records[ridx]['end_idx'] = to_i
                            range_records[ridx]['status'] = RangeStatus.Closed
                            ridx += 1
                else:
                    for j, from_i in enumerate(a_idxs):
                        valid_b_idxs = b_idxs[b_idxs >= from_i]
                        if len(valid_b_idxs) > 0:
                            to_i = valid_b_idxs[0]  # succeeding in b
                            range_records[ridx]['id'] = ridx
                            range_records[ridx]['col'] = col
                            range_records[ridx]['start_idx'] = from_i
                            range_records[ridx]['end_idx'] = to_i
                            range_records[ridx]['status'] = RangeStatus.Closed
                            ridx += 1
    return range_records[:ridx]

```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import between_two_ranges_nb

entries = np.array([[True, False, False, True, False]]).T
exits = np.array([[False, True, False, False, True]]).T

# 分析入场到退出的持仓时间
entry_to_exit = between_two_ranges_nb(entries, exits, from_other=False)
print(entry_to_exit)

## partition_ranges_nb
记录布尔矩阵 `a` 中的每组连续的 `True`：
- 遍历 `a` 的每一列，找到每组即在 `[start, end)` 上连续为 `True`，记录
  - `id`：被找到的顺序
  - `col`：所在列
  - `start_idx`：`start`
  - `end_idx`：`end`
  - `status`：`RangeStatus.Closed`

### 源码
```python
@njit(cache=True)
def partition_ranges_nb(a: tp.Array2d) -> tp.RecordArray:

    range_records = np.empty(a.shape[0] * a.shape[1], dtype=range_dt)
    ridx = 0

    for col in range(a.shape[1]):
        is_partition = False
        from_i = -1
        for i in range(a.shape[0]):
            if a[i, col]:
                if not is_partition:
                    from_i = i
                is_partition = True
            elif is_partition:
                to_i = i
                range_records[ridx]['id'] = ridx
                range_records[ridx]['col'] = col
                range_records[ridx]['start_idx'] = from_i
                range_records[ridx]['end_idx'] = to_i
                range_records[ridx]['status'] = RangeStatus.Closed
                ridx += 1
                is_partition = False
            if i == a.shape[0] - 1:
                if is_partition:
                    to_i = a.shape[0] - 1
                    range_records[ridx]['id'] = ridx
                    range_records[ridx]['col'] = col
                    range_records[ridx]['start_idx'] = from_i
                    range_records[ridx]['end_idx'] = to_i
                    range_records[ridx]['status'] = RangeStatus.Open
                    ridx += 1
    return range_records[:ridx]
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import partition_ranges_nb

signals = np.array([
    [False, True, True, False, False, True, True, True, False],
    [True, True, False, False, True, False, False, False, False]
]).T

# 分析信号分区
partitions = partition_ranges_nb(signals)
print(partitions)


## between_partition_ranges_nb
记录布尔矩阵 `a` 中相邻的两组连续 `True`：
- 遍历 `a` 的每一列，找到相邻的两组连续 `True`，记录
  - `id`：被找到的顺序
  - `col`：所在列
  - `start_idx`：位于前方的连续 `True` 的尾部索引
  - `end_idx`：位于后方的连续 `True` 的首部索引
  - `status`：`RangeStatus.Closed`

### 源码
```python
@njit(cache=True)
def between_partition_ranges_nb(a: tp.Array2d) -> tp.RecordArray:

    range_records = np.empty(a.shape[0] * a.shape[1], dtype=range_dt)
    ridx = 0

    for col in range(a.shape[1]):
        is_partition = False
        from_i = -1
        for i in range(a.shape[0]):
            if a[i, col]:
                if not is_partition and from_i != -1:
                    to_i = i
                    range_records[ridx]['id'] = ridx
                    range_records[ridx]['col'] = col
                    range_records[ridx]['start_idx'] = from_i
                    range_records[ridx]['end_idx'] = to_i
                    range_records[ridx]['status'] = RangeStatus.Closed
                    ridx += 1
                is_partition = True
                from_i = i
            else:
                is_partition = False
    return range_records[:ridx]
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import between_partition_ranges_nb

# 创建包含多个分区的信号序列
signals = np.array([
    [True, True, False, False, False, True, False, True, True],
    [False, True, True, False, True, True, False, False, True]
]).T

# 分析分区间的间隔
intervals = between_partition_ranges_nb(signals)
print(intervals)

# Ranking

## rank_nb
找到 `a` 的所有连续 `True` 组
- `after_false=True`：忽略开头没有 `False` 的连续 `True` 组
- `rank_func_nb`
  - 负责给每个连续 `True` 组内的索引分配一个自定义等级
  - 接收参数 `(i, col, reset_i, prev_part_end_i, part_start_i, *args)`
  - 返回 -1 表示不分配等级，返回 >=0 表示分配的等级
- `reset_by`
  - 如果 `reset_by[i, col]=True`，则 `rank_func_nb` 处理 `(i, col)` 传入重置信号

```python
@njit
def rank_nb(a: tp.Array2d,
            reset_by: tp.Optional[tp.Array1d],
            after_false: bool,
            rank_func_nb: tp.RankFunc, *args) -> tp.Array2d:

    out = np.full(a.shape, -1, dtype=np.int64)

    for col in range(a.shape[1]):
        reset_i = 0
        prev_part_end_i = -1
        part_start_i = -1
        in_partition = False
        false_seen = not after_false
        for i in range(a.shape[0]):
            if reset_by is not None:
                if reset_by[i, col]:
                    reset_i = i
            if a[i, col] and not (after_false and not false_seen):
                if not in_partition:
                    part_start_i = i
                # 分区开始
                in_partition = True
                out[i, col] = rank_func_nb(i, col, reset_i, prev_part_end_i, part_start_i, *args)
            elif not a[i, col]:
                if in_partition:
                # 分区结束
                    prev_part_end_i = i - 1
                in_partition = False
                false_seen = True
    return out
```

### 例子

In [ ]:
import numpy as np
from numba import njit
from vectorbt.signals.nb import rank_nb

@njit
def position_rank_func(i, col, reset_i, prev_part_end_i, part_start_i):
    # 简单的位置排序：分区内的位置索引
    return i - part_start_i

# 创建测试信号
signals = np.array([
    [False, True, True, False, True, True, True],
    [True, True, False, False, True, False, True]
]).T

# 为信号分配位置等级
ranks = rank_nb(signals, None, False, position_rank_func)
print(ranks)

## sig_pos_rank_nb
供函数 `rank_nb` 使用，按照信号在分区内的位置分配等级

参数
- `i` (int)：当前行索引
- `col` (int)：当前列索引
- `reset_i` (int): 最近重置信号的索引
- `prev_part_end_i` (int)：前一个分区的结束索引
- `part_start_i` (int)：当前分区的开始索引
- `sig_pos_temp` (np.array)：临时数组，用于维护每列的位置计数
- `allow_gaps` (bool)：是否允许位置间隔
  - True：连续计数，忽略分区内的False值
  - False：严格按分区边界重置计数

返回值：int，信号在分区内的位置等级，从 0 开始

```python
@njit(cache=True)
def sig_pos_rank_nb(i: int, col: int, reset_i: int, prev_part_end_i: int, part_start_i: int,
                    sig_pos_temp: tp.Array1d, allow_gaps: bool) -> int:

    if reset_i > prev_part_end_i and max(reset_i, part_start_i) == i:
        # 重置条件：有重置信号且在分区开始
        sig_pos_temp[col] = -1
    elif not allow_gaps and part_start_i == i:
        # 严格模式：分区开始时重置
        sig_pos_temp[col] = -1
    # 递增位置计数
    sig_pos_temp[col] += 1
    return sig_pos_temp[col]
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import rank_nb, sig_pos_rank_nb

# 为信号分配分区内位置等级
signals = np.array([[False, True, True, False, True, True]]).T
temp_array = np.zeros(1, dtype=np.int64)

# 使用严格分区计数
pos_ranks = rank_nb(signals, None, False, sig_pos_rank_nb, temp_array, False)
print(pos_ranks)

# 使用连续计数
pos_ranks = rank_nb(signals, None, False, sig_pos_rank_nb, temp_array, True)
print(pos_ranks)

## part_pos_rank_nb
供函数 `rank_nb` 使用，按照分区在序列中的位置分配等级

```python
@njit(cache=True)
def part_pos_rank_nb(i: int, col: int, reset_i: int, prev_part_end_i: int, part_start_i: int,
                     part_pos_temp: tp.Array1d) -> int:

    if reset_i > prev_part_end_i and max(reset_i, part_start_i) == i:
        part_pos_temp[col] = 0
    elif part_start_i == i:
        part_pos_temp[col] += 1
    return part_pos_temp[col]
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import rank_nb, part_pos_rank_nb

# 为信号分配分区等级
signals = np.array([[True, True, False, True, False, True, True]]).T
temp_array = np.zeros(1, dtype=np.int64)

# 使用分区位置排序
part_ranks = rank_nb(signals, None, False, part_pos_rank_nb, temp_array)
print(part_ranks)

# Index

## nth_index_1d_nb
获取一维数组 `a` 中第 `n` 个 True 值的索引位置

参数
- `a` (np.array)：一维布尔数组
- `n` (int)：目标True值的序号
  - `n>=0`：正向索引，0表示第1个True值
  - `n<0`：反向索引，-1表示最后1个True值

返回：int，第 n 个 True 值的索引位置，如果不存在则返回 -1

### 源码
```python
@njit(cache=True)
def nth_index_1d_nb(a: tp.Array1d, n: int) -> int:

    if n >= 0:
        found = -1
        for i in range(a.shape[0]):
            if a[i]:
                found += 1
                if found == n:
                    return i
    else:
        found = 0
        for i in range(a.shape[0] - 1, -1, -1):
            if a[i]:
                found -= 1
                if found == n:
                    return i
    return -1
```

### 例子

In [ ]:
from vectorbt.signals.nb import nth_index_1d_nb

empty_signals = np.array([False, False, False, False, False, False, False, False, False, False])
single_signal = np.array([False, True, False, True, True, False, True, False, False, True])

result = nth_index_1d_nb(empty_signals, 0)
print(result)
result = nth_index_1d_nb(single_signal, 5)
print(result)
result = nth_index_1d_nb(single_signal, -5)
print(result)
result = nth_index_1d_nb(single_signal, 0) 
print(result)
result = nth_index_1d_nb(single_signal, -1)
print(result)

## nth_index_nb
二维版本的 `nth_index_nb`

### 源码
```python
@njit(cache=True)
def nth_index_nb(a: tp.Array2d, n: int) -> tp.Array1d:

    out = np.empty(a.shape[1], dtype=np.int64)
    for col in range(a.shape[1]):
        out[col] = nth_index_1d_nb(a[:, col], n)
    return out
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import nth_index_nb

# 创建多资产信号矩阵
signals = np.array([
    [False, True, False],   # 时间0: 只有资产1有信号
    [True, False, True],    # 时间1: 资产0和2有信号  
    [False, True, False],   # 时间2: 只有资产1有信号
    [True, False, True],    # 时间3: 资产0和2有信号
    [False, False, True]    # 时间4: 只有资产2有信号
])

# 查找每个资产的第1个信号位置
first_signals = nth_index_nb(signals, 0)
print(f"各资产首次信号位置: {first_signals}")  # [1, 0, 1]

# 查找每个资产的第2个信号位置  
second_signals = nth_index_nb(signals, 1)
print(f"各资产第2次信号位置: {second_signals}")  # [3, 2, 3]

# 查找每个资产的最后一个信号位置
last_signals = nth_index_nb(signals, -1)
print(f"各资产最后信号位置: {last_signals}")  # [3, 2, 4]

## norm_avg_index_1d_nb
计算布尔数组 `a` 中所有 True 值的索引的平均值，并相对于数组长度归一化到 [-1, 1] 区间

### 源码
```python
@njit(cache=True)
def norm_avg_index_1d_nb(a: tp.Array1d) -> float:

    mean_index = np.mean(np.flatnonzero(a))
    return renormalize_nb(mean_index, (0, len(a) - 1), (-1, 1))
```

### 例子

In [ ]:
import numpy as np

from vectorbt.signals.nb import norm_avg_index_1d_nb

# 假设有10天的信号，True表示当天有信号
signals = np.array([True, False, False, True, False, False, False, False, False, False])
# True信号出现在第0天和第3天，平均索引为(0+3)/2=1.5
# 归一化到[-1, 1]区间：mean_index=1.5, 原区间[0,9]，归一化后为 -0.6667
result = norm_avg_index_1d_nb(signals)
print(result)  # 输出约为 -0.6667，说明信号偏前

signals2 = np.array([False, False, False, False, False, False, False, True, True, True])
# True信号出现在第7、8、9天，平均索引为(7+8+9)/3=8
# 归一化后为 0.7778，说明信号偏后
print(norm_avg_index_1d_nb(signals2))

## norm_avg_index_nb
二维版本的 `norm_avg_index_1d_nb`

### 源码
```python
@njit(cache=True)
def norm_avg_index_nb(a: tp.Array2d) -> tp.Array1d:

    out = np.empty(a.shape[1], dtype=np.float64)
    for col in range(a.shape[1]):
        out[col] = norm_avg_index_1d_nb(a[:, col])
    return out
```

### 例子

In [ ]:
import numpy as np
from vectorbt.signals.nb import norm_avg_index_nb

# 三只股票，10天的信号
signals = np.array([
    [True,  False, False],
    [False, False, True ],
    [False, True,  False],
    [True,  False, False],
    [False, False, False],
    [False, False, False],
    [False, False, False],
    [False, True,  True ],
    [False, False, False],
    [False, False, True ]
])

# 计算每只股票信号的归一化平均索引
result = norm_avg_index_nb(signals)
print(result)
# 输出如: [-0.6667, 0.3333, 0.7778]
# 说明第一只股票信号偏前，第二只偏中，第三只偏后